In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('ggplot')
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, classification_report

%pip install xgboost
%pip install lightgbm

# Dataset Extraction

In [ ]:
df = pd.read_csv('Premier_League.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

# Exploratory Data Analysis

In [ ]:
# Home Goal Distribution
plt.figure(figsize = (10, 5))
sns.histplot(df['Goals Home'], bins = 10, kde = True)
plt.title('Home Goals Distribution')
plt.xlabel('Goals Score by Home Team')
plt.ylabel('Count')
plt.grid(True)
plt.show()

In [ ]:
# Away Goals Distribution
plt.figure(figsize = (10, 5))
sns.histplot(df['Away Goals'], bins = 10, kde = True)
plt.title('Away Goals Distribution')
plt.xlabel('Goals Score by Away Team')
plt.ylabel('Count')
plt.grid(True)
plt.show()

In [ ]:
# Distribution of Home and Away Goals
plt.figure(figsize = (10, 5))
sns.histplot(df['Goals Home'], bins = 10, kde = True, color = 'blue', label = 'Home Goals')
sns.histplot(df['Away Goals'], bins = 10, kde = True, color = 'red', label = 'Away Goals')
plt.title('Home Goals & Away Goals Distribution')
plt.xlabel('Goals Scored')
plt.ylabel('Count')
plt.legend()
plt.grid(True)
plt.show()

Histogram shows that home team makes the most goal throughout the year

In [ ]:
# Attendance Variation over time
df['date'] = pd.to_datetime(df['date'], format='mixed')
df['attendance'] = df['attendance'].astype(str).str.replace(',', '').str.replace('Nan', '0').astype(int) # Take out the , from the attendance and convert to integer
attendance_overtime = df.groupby('date')['attendance'].sum()

plt.figure(figsize = (10, 5))
attendance_overtime.plot(kind = 'line', marker = 'o')
plt.title('Attendance Variation Over Time')
plt.xlabel('Date')
plt.ylabel('Total Attendance')
plt.grid(True)
plt.show()

Attendance peaked between 2022-11 & 2022-12. The highest peaked at the end of 2023-06

In [ ]:
# Comparison of Home and Away possessions
plt.figure(figsize = (10, 6))
sns.boxplot(df[['home_possessions', 'away_possessions']])
plt.title('Comparison of Home and Away possessions')
plt.xlabel('Teams')
plt.ylabel('Possessions (%)')
plt.xticks(ticks = [0, 1], labels = ['Home Team', 'Away Team'])
plt.grid(True)
plt.show()

In [ ]:
# Comparing Home and Away Shot Accuracy
df['home_shot_accuracy'] = df['home_on'] / df['home_shots']
df['away_shot_accuracy'] = df['away_on'] / df['away_shots']

plt.figure(figsize = (10, 5))
sns.barplot(x = ['Home Team', 'Away Team'], y = [df['home_shot_accuracy'].mean(), df['away_shot_accuracy'].mean()])
plt.title('Comparison of Home and Away Shot Accuracy')
plt.xlabel('Teams')
plt.ylabel('Shot Accuracy')
plt.grid(True)
plt.show()

# Feature Engineering

### Data cleaning & Label Encoding

In [ ]:
# Drop irrelevant columns
df = df.drop(columns=['clock', 'stadium', 'attendance', 'links'])

In [ ]:
# Convert date to numerical feature
df['date'] = pd.to_datetime(df['date'])
df['day_of_year'] = df['date'].dt.dayofyear

In [ ]:
# Calculate goal difference and home team win percentage
df['goal_difference'] = df['Goals Home'] - df['Away Goals']
df['home_team_win'] = df['goal_difference'].apply(lambda x: 1 if x > 0 else 0)
df['result'] = np.select(
    [df['goal_difference'] > 0, df['goal_difference'] == 0],
    [1, 0],
    default=-1)

In [ ]:
# Encode Team Names to categorical columns
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df['Home Team'] = label_encoder.fit_transform(df['Home Team'])
df['Away Team'] = label_encoder.fit_transform(df['Away Team'])

In [ ]:
df.head()

### Rolling Expected Goals (last 5 seasons)

In [ ]:
# An approximate expected goals
df['home_xG_proxy'] = df['home_on'] * 0.15 + df['home_off'] * 0.05
df['away_xG_proxy'] = df['away_on'] * 0.15 + df['away_off'] * 0.05

In [ ]:
# Home Team data
home = df[['date', 'Home Team', 'home_xG_proxy']].copy()
home.columns = ['date', 'Team', 'xG']

# Away Team data
away = df[['date', 'Away Team', 'away_xG_proxy']].copy()
away.columns = ['date', 'Team', 'xG']

team_df = pd.concat([home, away])

In [ ]:
team_df = team_df.sort_values(by = 'date')

In [ ]:
# 1. Reset the index to make it unique
team_df = team_df.reset_index(drop=True)

# 2. Run your original code
team_df["rolling_avg_xG"] = (
    team_df.groupby("Team")["xG"]
    .rolling(window=5)
    .mean()
    .reset_index(level=0, drop=True)
)

In [ ]:
team_df

In [ ]:
df = df.merge(team_df[['date', 'Team', 'rolling_avg_xG']], left_on=['date', 'Home Team'], right_on=['date', 'Team'], how='left').rename(columns={'rolling_avg_xG': 'home_rolling_avg_xG'}).drop(columns=['Team'])

df = df.merge(team_df[['date', 'Team', 'rolling_avg_xG']], left_on=['date', 'Away Team'], right_on=['date', 'Team'], how='left').rename(columns={'rolling_avg_xG': 'away_rolling_avg_xG'}).drop(columns=['Team'])

In [ ]:
df.dropna(inplace = True)

### Rolling Average for possessions (last 5 seasons)

In [ ]:
# Home Team data
home = df[['date', 'Home Team', 'home_possessions']].copy()
home.columns = ['date', 'Team', 'Possessions']

# Away Team data
away = df[['date', 'Away Team', 'away_possessions']].copy()
away.columns = ['date', 'Team', 'Possessions']

team_df = pd.concat([home, away])

In [ ]:
team_df = team_df.sort_values(by = 'date')

In [ ]:
# 1. Reset the index to make it unique
team_df = team_df.reset_index(drop=True)

# 2. Run your original code
team_df["rolling_avg_possessions"] = (
    team_df.groupby("Team")["Possessions"]
    .rolling(window=5)
    .mean()
    .reset_index(level=0, drop=True)
)

In [ ]:
team_df

Empty rolling averages present. Remove them.

Merge RA to original dataset

In [ ]:
df = df.merge(team_df[['date', 'Team', 'rolling_avg_possessions']], left_on=['date', 'Home Team'], right_on=['date', 'Team'], how='left').rename(columns={'rolling_avg_possessions': 'home_rolling_avg_possessions'}).drop(columns=['Team'])

In [ ]:
df = df.merge(team_df[['date', 'Team', 'rolling_avg_possessions']], left_on=['date', 'Away Team'], right_on=['date', 'Team'], how='left').rename(columns={'rolling_avg_possessions': 'away_rolling_avg_possessions'}).drop(columns=['Team'])

In [ ]:
df

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace = True)

### Rolling Average for shot accuracy (last 5 seasons)

In [ ]:
# Home Team data
home = df[['date', 'Home Team', 'home_shot_accuracy']].copy()
home.columns = ['date', 'Team', 'Shot Accuracy']

# Away Team data
away = df[['date', 'Away Team', 'away_shot_accuracy']].copy()
away.columns = ['date', 'Team', 'Shot Accuracy']

team_df = pd.concat([home, away])

In [ ]:
team_df = team_df.sort_values(by = 'date')

In [ ]:
# 1. Reset the index to make it unique
team_df = team_df.reset_index(drop=True)

# 2. Run your original code
team_df["rolling_avg_shot_accuracy"] = (
    team_df.groupby("Team")["Shot Accuracy"]
    .rolling(window=5)
    .mean()
    .reset_index(level=0, drop=True)
)

In [ ]:
team_df

Empty rolling averages present. Remove them.

Merge RA to original dataset

In [ ]:
df = df.merge(team_df[['date', 'Team', 'rolling_avg_shot_accuracy']], left_on=['date', 'Home Team'], right_on=['date', 'Team'], how='left').rename(columns={'rolling_avg_shot_accuracy': 'home_rolling_avg_shot_accuracy'}).drop(columns=['Team'])

In [ ]:
df = df.merge(team_df[['date', 'Team', 'rolling_avg_shot_accuracy']], left_on=['date', 'Away Team'], right_on=['date', 'Team'], how='left').rename(columns={'rolling_avg_shot_accuracy': 'away_rolling_avg_shot_accuracy'}).drop(columns=['Team'])

In [ ]:
df

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace = True)

### Rolling Average for chances (last 5 seasons)

In [ ]:
# Home Team data
home = df[['date', 'Home Team', 'home_chances']].copy()
home.columns = ['date', 'Team', 'Chances']

# Away Team data
away = df[['date', 'Away Team', 'away_chances']].copy()
away.columns = ['date', 'Team', 'Chances']

team_df = pd.concat([home, away])

In [ ]:
team_df = team_df.sort_values(by = 'date')

In [ ]:
# 1. Reset the index to make it unique
team_df = team_df.reset_index(drop=True)

# 2. Run your original code
team_df["rolling_avg_chances"] = (
    team_df.groupby("Team")["Chances"]
    .rolling(window=5)
    .mean()
    .reset_index(level=0, drop=True)
)

In [ ]:
team_df

Empty rolling averages present. Remove them.

Merge RA to original dataset

In [ ]:
df = df.merge(team_df[['date', 'Team', 'rolling_avg_chances']], left_on=['date', 'Home Team'], right_on=['date', 'Team'], how='left').rename(columns={'rolling_avg_chances': 'home_rolling_avg_chances'}).drop(columns=['Team'])

In [ ]:
df = df.merge(team_df[['date', 'Team', 'rolling_avg_chances']], left_on=['date', 'Away Team'], right_on=['date', 'Team'], how='left').rename(columns={'rolling_avg_chances': 'away_rolling_avg_chances'}).drop(columns=['Team'])

In [ ]:
df

In [ ]:
df.dropna(inplace = True)

### Rolling Average for goals (last 5 seasons)

In [ ]:
# Home Team data
home = df[['date', 'Home Team', 'Goals Home']].copy()
home.columns = ['date', 'Team', 'Goals']

# Away Team data
away = df[['date', 'Away Team', 'Away Goals']].copy()
away.columns = ['date', 'Team', 'Goals']

team_df = pd.concat([home, away])

In [ ]:
team_df = team_df.sort_values(by = 'date')

In [ ]:
# 1. Reset the index to make it unique
team_df = team_df.reset_index(drop=True)

# 2. Run your original code
team_df["rolling_avg_goals"] = (
    team_df.groupby("Team")["Goals"]
    .rolling(window=5)
    .mean()
    .reset_index(level=0, drop=True)
)

In [ ]:
team_df

Empty rolling averages present. Remove them.

Merge RA to original dataset

In [ ]:
df = df.merge(team_df[['date', 'Team', 'rolling_avg_goals']], left_on=['date', 'Home Team'], right_on=['date', 'Team'], how='left').rename(columns={'rolling_avg_goals': 'home_rolling_avg_goals'}).drop(columns=['Team'])

In [ ]:
df = df.merge(team_df[['date', 'Team', 'rolling_avg_goals']], left_on=['date', 'Away Team'], right_on=['date', 'Team'], how='left').rename(columns={'rolling_avg_goals': 'away_rolling_avg_goals'}).drop(columns=['Team'])

In [ ]:
df.dropna(inplace = True)

In [ ]:
df

# Data Splitting

In [ ]:
# 1. Manually select only 'Pre-Match' features
df['avg_shot_diff'] = df['home_rolling_avg_shot_accuracy'] - df['away_rolling_avg_shot_accuracy']
df['avg_goal_diff'] = df['home_rolling_avg_goals'] - df['away_rolling_avg_goals']
df['avg_possess_diff'] = df['home_rolling_avg_possessions'] - df['away_rolling_avg_possessions']
df['home_goal_chances'] = np.where(df['home_rolling_avg_chances'] == 0, 0,df['home_rolling_avg_goals'] / df['home_rolling_avg_chances'])
df['away_goal_chances'] = np.where(df['away_rolling_avg_chances'] == 0, 0, df['away_rolling_avg_goals'] / df['away_rolling_avg_chances'])


predictors = ['Home Team', 'Away Team', 'avg_shot_diff', 'avg_goal_diff', 'avg_possess_diff', 'home_goal_chances', 'away_goal_chances', 'home_rolling_avg_xG', 'away_rolling_avg_xG']  # Data before whistle blows / before match ends
X = df[predictors]
y = df['home_team_win']

# Chronological Split (No random shuffling)
# Assuming your df is sorted by date
# split_index = int(len(df) * 0.8)
# X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
# y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# Time Series Splitting
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
for train_index, test_index in tscv.split(df):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
eval_set = [(X_train, y_train), (X_test, y_test)]

# Random Forest Classifier

In [ ]:
clf = RandomForestClassifier(min_samples_leaf=2, min_samples_split=5, n_estimators=200, bootstrap=True)
clf.fit(X_train, y_train)

In [ ]:
rf_preds = clf.predict(X_test)

In [ ]:
acc = accuracy_score(y_test, rf_preds)
prec = precision_score(y_test, rf_preds)

print(f"Accuracy: {acc:.2%}")
print(f"Precision: {prec:.2%}")

In [ ]:
print(classification_report(y_test, rf_preds))

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}

gs = GridSearchCV(clf, param_grid, scoring='neg_mean_squared_error', cv = 3)
gs.fit(X_train, y_train)

In [ ]:
print(gs.best_params_)

# XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

xgb = XGBClassifier(
    subsample=0.8,
    n_estimators=100,
    max_depth=7,
    learning_rate=0.1,
    colsample_bytree=1.0,
    reg_alpha=5.0,
    reg_lambda=10.0,
    early_stopping_rounds = 20
    )

xgb.fit(X_train, y_train, eval_set=eval_set, verbose=False)

In [ ]:
param_grid_xgb = {
    'max_depth': [6, 7, 8, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'n_estimators': [100, 500, 1000]
}

gs = GridSearchCV(xgb, param_grid_xgb, scoring='neg_mean_squared_error', cv = 5)
gs.fit(X_train, y_train, eval_set=eval_set, verbose=False)

In [ ]:
print(gs.best_params_)

In [ ]:
results = xgb.evals_result()
print(results.keys())

train_loss = results['validation_0']['logloss']
val_loss = results['validation_1']['logloss']

In [ ]:
xgb_pred = xgb.predict(X_test)

acc = accuracy_score(y_test, xgb_pred)
prec = precision_score(y_test, xgb_pred)

print(f"Accuracy: {acc:.2%}")
print(f"Precision: {prec:.2%}")

In [ ]:
print(classification_report(y_test, xgb_pred))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.plot(train_loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.xlabel('Boosting Round')
plt.ylabel('Log Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.show()


# LightGBM

In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(
    colsample_bytree=0.6,
    max_depth=6,
    learning_rate=0.1,
    n_estimators=500,
    num_leaves=31,
    subsample=0.6,
    lambda_l1=8.0,
    lambda_l2=15.0,
    min_child_weight=5,
    min_child_samples=20,
    feature_fraction=0.7,
    bagging_fraction=0.7,
    verbose = -1
    )

lgbm.fit(X_train, y_train, eval_set=eval_set, eval_metric='binary_logloss')

In [ ]:
results = lgbm.evals_result_
print(results.keys())

train_loss = results['training']['binary_logloss']
val_loss = results['valid_1']['binary_logloss']

In [ ]:
lgbm_pred = lgbm.predict(X_test)

acc = accuracy_score(y_test, lgbm_pred)
prec = precision_score(y_test, lgbm_pred)

print(f"Accuracy: {acc:.2%}")
print(f"Precision: {prec:.2%}")

In [ ]:
print(classification_report(y_test, lgbm_pred))

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(train_loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.xlabel('Boosting Round')
plt.ylabel('Log Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.show()

### GridSearchCV

In [ ]:
param_grid = {
    'max_depth': [6, 7, 8, 9],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 500, 1000],
    'min_child_samples': [20, 30, 35]
}

gs = GridSearchCV(lgbm, param_grid, scoring='neg_mean_squared_error', cv = 5, verbose=0)
gs.fit(X_train, y_train)

In [ ]:
print(gs.best_params_)

# Model Testing

In [216]:
# Assuming your classes are ordered [Away Win (-1), Draw (0), Home Win (1)]

test_set = pd.DataFrame({'Actual': y_test, 'LightGBM': lgbm_pred, 'XGBoost': xgb_pred, 'Random Forest': rf_preds})

# Select a random row from the test set
sample_input = X_test.sample(n=1, random_state=42)

actual_outcome = y_test.loc[sample_input.index].values[0]

# Get the probability for each classifier
lgbm_prediction = lgbm.predict_proba(sample_input)[0]
xgb_prediction = xgb.predict_proba(sample_input)[0]
rf_prediction = clf.predict_proba(sample_input)[0]

# Convert the encoded team IDs back to team names
home_team_name = label_encoder.inverse_transform(sample_input['Home Team'].values.reshape(1))[0]
away_team_name = label_encoder.inverse_transform(sample_input['Away Team'].values.reshape(1))[0]

In [218]:
classes = label_encoder.classes_

def format_probs(probs):
    return {classes[i]: f"{p:.2%}" for i, p in enumerate(probs)}

print(f"Match Analysis: {home_team_name} vs {away_team_name}")
print(f"Actual Outcome: {actual_outcome}")
print("-" * 30)
print(f"Random Forest Confidence: {format_probs(rf_prediction)}")
print(f"LightGBM Confidence: {format_probs(lgbm_prediction)}")
print(f"XGBoost Confidence: {format_probs(xgb_prediction)}")

Match Analysis: Aston Villa vs Arsenal
Actual Outcome: 0
------------------------------
Random Forest Confidence: {'Arsenal': '60.59%', 'Aston Villa': '39.41%'}
LightGBM Confidence: {'Arsenal': '58.92%', 'Aston Villa': '41.08%'}
XGBoost Confidence: {'Arsenal': '61.61%', 'Aston Villa': '38.39%'}
